In [4]:
%pip install nflreadpy

  Using cached nflreadpy-0.1.5-py3-none-any.whl.metadata (7.5 kB)
  Using cached polars-1.44.1-py3-none-any.whl.metadata (11 kB)
  Using cached polars_runtime_32-1.44.1-cp310-abi3-macosx_11_0_arm64.whl.metadata (1.5 kB)
Using cached nflreadpy-0.1.5-py3-none-any.whl (30 kB)
Using cached polars-1.44.1-py3-none-any.whl (865 kB)
Using cached polars_runtime_32-1.44.1-cp310-abi3-macosx_11_0_arm64.whl (48.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [nflreadpy]/3 [polars]
Note: you may need to restart the kernel to use updated packages.


In [12]:
import nflreadpy as nfl
import pandas as pd

pbp = nfl.load_pbp(list(range(2015, 2025)))  # 10 seasons
pbp = pbp.to_pandas()

In [7]:
fourth = pbp[pbp['down'] == 4].copy()

features = fourth[[
    'yardline_100',       # field position (distance from opponent goal)
    'ydstogo',            # distance to go
    'score_differential', # offense - defense
    'game_seconds_remaining',
    'qtr',
]].copy()

# Label the actual decision
def label_decision(row):
    if row['play_type'] == 'punt':
        return 'punt'
    elif row['play_type'] == 'field_goal':
        return 'field_goal'
    elif row['play_type'] in ('run', 'pass'):
        return 'go'
    return None

fourth['decision'] = fourth.apply(label_decision, axis=1)
fourth = fourth.dropna(subset=['decision'])

In [8]:
fourth = fourth[['yardline_100', 'ydstogo', 'score_differential',
                  'game_seconds_remaining', 'qtr', 'decision', 'epa']].copy()

In [9]:
fourth.isna().sum()          # see how much is missing, column by column
fourth = fourth.dropna()     # drop rows with any missing values

In [10]:
fourth['decision'].value_counts()

decision
punt          23028
field_goal     9718
go             6629
Name: count, dtype: int64

In [17]:
# converting descision into a format a neural netowrk can read
decision_dummies = pd.get_dummies(fourth['decision'])
fourth = pd.concat([fourth, decision_dummies], axis=1)

In [14]:
fourth.to_parquet('cleaned_fourth_downs.parquet')